# MovieLens 32M — Ingestion
_Phase 01 — Docker cluster — écrit dans `data/warehouse/movielens_*/`_

## Prérequis
Télécharger le dataset sur https://grouplens.org/datasets/movielens/ → **ml-32m.zip**  
Dézipper dans `data/raw/MOVIELENS/ml-32m/` — le dossier doit contenir :
```
data/raw/MOVIELENS/ml-32m/
├── movies.csv      (~1.4 Mo)
├── ratings.csv     (~900 Mo)
└── links.csv       (~1.4 Mo)
```

## Output
```
data/warehouse/movielens_movies/     ← catalogue films
data/warehouse/movielens_ratings/    ← 32M notes
data/warehouse/movielens_links/      ← movieId ↔ tmdbId (pour affiches TMDB)
```

In [1]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, LongType, FloatType, StringType
)

spark = SparkSession.builder \
    .appName("MyDigitalTwin - MovieLens32M Ingestion") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(_os.path.join(_os.path.dirname('__file__'), '../../..')))
from config import WAREHOUSE, RAW_DATA

ML_DIR = os.path.join(RAW_DATA, "MOVIELENS", "ml-32m")
assert os.path.exists(ML_DIR), (
    f"Dossier introuvable : {ML_DIR}\n"
    "Télécharger ml-32m.zip sur grouplens.org et dézipper dans data/raw/MOVIELENS/"
)
print(f"Source ML : {ML_DIR}")
print(f"Warehouse  : {WAREHOUSE}")

Spark version : 3.5.5
Source ML : /opt/spark/data/raw/MOVIELENS/ml-32m
Warehouse  : /opt/spark/data/warehouse


26/04/15 10:11:25 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# ── 1. MOVIES — catalogue mondial ─────────────────────────────────────────────
# Format CSV : movieId,title,genres
# title = "Toy Story (1995)", genres = "Adventure|Animation|..."

movies_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("title",   StringType(),  False),
    StructField("genres",  StringType(),  True),
])

movies = spark.read \
    .option("header", "true") \
    .schema(movies_schema) \
    .csv(os.path.join(ML_DIR, "movies.csv"))

# Titre normalisé (sans année) — Column expression native, pas de UDF
movies = movies.withColumn(
    "title_norm",
    F.regexp_replace(
        F.regexp_replace(F.lower(F.col("title")), r'\(\d{4}\)', ''),
        r'[^\w\s]', ''
    )
).withColumn("title_norm", F.trim(F.col("title_norm")))

n_movies = movies.count()
print(f"Films dans MovieLens : {n_movies:,}")
movies.show(5, truncate=50)

Films dans MovieLens : 87,585
+-------+----------------------------------+-------------------------------------------+---------------------------+
|movieId|                             title|                                     genres|                 title_norm|
+-------+----------------------------------+-------------------------------------------+---------------------------+
|      1|                  Toy Story (1995)|Adventure|Animation|Children|Comedy|Fantasy|                  toy story|
|      2|                    Jumanji (1995)|                 Adventure|Children|Fantasy|                    jumanji|
|      3|           Grumpier Old Men (1995)|                             Comedy|Romance|           grumpier old men|
|      4|          Waiting to Exhale (1995)|                       Comedy|Drama|Romance|          waiting to exhale|
|      5|Father of the Bride Part II (1995)|                                     Comedy|father of the bride part ii|
+-------+-------------------------

In [3]:
# ── 2. MOVIES → warehouse/movielens_movies/ ───────────────────────────────────

out_movies = os.path.join(WAREHOUSE, "movielens_movies")
movies.write.mode("overwrite").parquet(out_movies)
print(f"Écrit : {out_movies}  ({n_movies:,} films)")

Écrit : /opt/spark/data/warehouse/movielens_movies  (87,585 films)


In [4]:
# ── 3. RATINGS — 32M notes ────────────────────────────────────────────────────
# Format CSV : userId,movieId,rating,timestamp
# rating : 0.5 à 5.0 (demi-étoiles)

ratings_schema = StructType([
    StructField("userId",    IntegerType(), False),
    StructField("movieId",   IntegerType(), False),
    StructField("rating",    FloatType(),   False),
    StructField("timestamp", LongType(),    True),
])

ratings = spark.read \
    .option("header", "true") \
    .schema(ratings_schema) \
    .csv(os.path.join(ML_DIR, "ratings.csv"))

n_ratings = ratings.count()
print(f"Notes totales : {n_ratings:,}")
ratings.show(5)

Notes totales : 32,000,204
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|     17|   4.0|944249077|
|     1|     25|   1.0|944250228|
|     1|     29|   2.0|943230976|
|     1|     30|   5.0|944249077|
|     1|     32|   5.0|943228858|
+------+-------+------+---------+
only showing top 5 rows



In [5]:
# ── 4. RATINGS → warehouse/movielens_ratings/ ────────────────────────────────
# Pas de repartition() : forcer un shuffle sur 32M lignes tue les executors (OOM).
# On écrit avec le partitionnement naturel issu de la lecture CSV.

out_ratings = os.path.join(WAREHOUSE, "movielens_ratings")
ratings.write.mode("overwrite").parquet(out_ratings)
print(f"Écrit : {out_ratings}  ({n_ratings:,} lignes)")

Écrit : /opt/spark/data/warehouse/movielens_ratings  (32,000,204 lignes)


In [6]:
# ── 5. LINKS — movieId ↔ tmdbId (pour affiches TMDB) ─────────────────────────
# Format CSV : movieId,imdbId,tmdbId
# tmdbId est utilisé pour fetcher posters + synopsis via l'API TMDB

links_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("imdbId",  StringType(),  True),
    StructField("tmdbId",  IntegerType(), True),
])

links = spark.read \
    .option("header", "true") \
    .schema(links_schema) \
    .csv(os.path.join(ML_DIR, "links.csv"))

n_links = links.count()
print(f"Liens : {n_links:,}")
links.show(5)

Liens : 87,585
+-------+-------+------+
|movieId| imdbId|tmdbId|
+-------+-------+------+
|      1|0114709|   862|
|      2|0113497|  8844|
|      3|0113228| 15602|
|      4|0114885| 31357|
|      5|0113041| 11862|
+-------+-------+------+
only showing top 5 rows



In [7]:
# ── 6. LINKS → warehouse/movielens_links/ ─────────────────────────────────────

out_links = os.path.join(WAREHOUSE, "movielens_links")
links.write.mode("overwrite").parquet(out_links)
print(f"Écrit : {out_links}  ({n_links:,} liens)")

Écrit : /opt/spark/data/warehouse/movielens_links  (87,585 liens)


In [8]:
# ── 7. VALIDATION ─────────────────────────────────────────────────────────────

for name, path in [
    ("movielens_movies",   out_movies),
    ("movielens_ratings",  out_ratings),
    ("movielens_links",    out_links),
]:
    df = spark.read.parquet(path)
    print(f"{name} : {df.count():,} lignes — schéma : {[f.name for f in df.schema.fields]}")

spark.stop()
print("\nIngestion MovieLens terminée. Lance 03_als/03_movie_recommendations.ipynb.")

movielens_movies : 87,585 lignes — schéma : ['movieId', 'title', 'genres', 'title_norm']
movielens_ratings : 32,000,204 lignes — schéma : ['userId', 'movieId', 'rating', 'timestamp']
movielens_links : 87,585 lignes — schéma : ['movieId', 'imdbId', 'tmdbId']

Ingestion MovieLens terminée. Lance 03_als/03_movie_recommendations.ipynb.
